# Experiment H21b: Does Layer Balance Predict When Muon's Directional Advantage Matters?

---

## Theoretical Background

Experiment H21a identified a crossover between Muon's "direction regime" (small $c$,
Muon wins) and the "scale regime" (large $c$, oracle per-layer SGD wins). This experiment
digs deeper into **why** the directional advantage diminishes with increasing imbalance.

### The Directional Value Hypothesis

Muon's Newton-Schulz step does two things simultaneously:
1. **Normalizes magnitude** (per-layer LR adaptation)
2. **Projects onto the orthogonal group** (directional quality / SV equalization)

To isolate the directional component, we compare Muon against **NormSGD** -- SGD where
each layer's gradient is divided by its Frobenius norm: $\hat{g}_\ell = g_\ell / \|g_\ell\|_F$.
NormSGD replicates Muon's scale normalization but uses the normalized gradient direction
instead of the polar factor. The ratio $\text{NormSGD loss} / \text{Muon loss}$ is
the **residual directional advantage** -- the benefit Muon gets purely from its
directional quality (SV equalization).

### Why Balance Matters

When layers are **balanced** ($c \approx 1$):
- All layer gradients have comparable magnitudes
- The singular value structure of each gradient carries meaningful information about
  the loss landscape geometry
- The polar factor's SV equalization effectively uses this structure

When layers are **imbalanced** ($c \gg 1$):
- Gradient magnitudes differ by orders of magnitude
- Small-gradient layers have their SV structure dominated by numerical noise
- The polar factor still equalizes SVs, but it is equalizing noise rather than signal
- Only the magnitude normalization provides value

### Prediction

The residual directional advantage should **monotonically decrease** with increasing $c$:
- At $c = 1$: advantage $\sim 5\text{-}20$x
- At $c > 50$: advantage $< 2$x (direction barely matters)

## Experimental Design

| Method | Magnitude normalization | Direction |
|:-------|:-----------------------|:----------|
| Muon | $\|M\|_F$ normalization in NS | Polar factor (orthogonal) |
| NormSGD | $\|g\|_F$ normalization | Normalized gradient (same direction as SGD) |

Both methods use single-LR sweeps and momentum. The ratio isolates direction vs. scale.

## Key Tests

| Test | Criterion | Meaning |
|:-----|:----------|:--------|
| T1 | $r < -0.5$ between $\log c$ and $\log(\text{adv})$ | Strong negative correlation: balance predicts directional value |
| T2 | $\text{adv}(c=1) > 3 \times \text{adv}(c=100)$ | Dramatic reduction in directional value with imbalance |

## Environment Setup

Import NumPy for linear algebra operations.

In [ ]:
import numpy as np
import os

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath('.'))


## Experimental Configuration

4-layer deep linear network with 32x32 matrices, 300 steps, momentum 0.9.
Five seeds for robust averaging. Balance ratios swept from $c = 1$ (perfect balance)
to $c = 100$ (extreme imbalance).

In [ ]:
DIM = 32
N_LAYERS = 4
NUM_STEPS = 300
MOMENTUM = 0.9
NS_ITERS = 5
NUM_SEEDS = 5
BATCH_SIZE = 64

print("\n--- Experimental Configuration ---")
print(f"  DIM = {DIM}")
print(f"  NUM_STEPS = {NUM_STEPS}")
print(f"  BATCH_SIZE = {BATCH_SIZE}")
print(f"  MOMENTUM = {MOMENTUM}")
print(f"  NUM_SEEDS = {NUM_SEEDS}")
print(f"  NS_ITERS = {NS_ITERS}")
print(f"  N_LAYERS = {N_LAYERS}")


In [ ]:
C_VALUES = [1.0, 1.5, 2.0, 3.0, 5.0, 10.0, 30.0, 100.0]

The $c$ values include fine resolution at the low end ($1, 1.5, 2, 3, 5$) to capture
the transition region where directional advantage begins to diminish, plus coarser
sampling at higher imbalance levels ($10, 30, 100$).

Learning rates are swept on log-uniform grids. Muon and NormSGD use different ranges
because their effective update scales differ (NormSGD produces unit-norm updates while
Muon's orthogonal updates have $\|U\|_F = \sqrt{d}$).

In [ ]:
LR_MUON = np.logspace(-4, -1, 10)
LR_NORM = np.logspace(-3, 0, 10)

## Core Algorithms

### Newton-Schulz Orthogonalization
Projects onto the nearest orthogonal matrix. This provides both scale normalization
AND directional quality (SV equalization).

### Weight Initialization with Rescaling
Near-identity plus $c$-rescaling on first/last layers, preserving the end-to-end product.

In [ ]:
def newton_schulz(M, n_iters=NS_ITERS):
    norm = np.linalg.norm(M, 'fro')
    if norm < 1e-15:
        return M
    X = M / norm
    for _ in range(n_iters):
        A = X.T @ X
        X = 1.5 * X - 0.5 * X @ A
    return X

In [ ]:
def init_weights(seed, c):
    rng = np.random.RandomState(seed)
    weights = [np.eye(DIM) + rng.randn(DIM, DIM) * 0.1 for _ in range(N_LAYERS)]
    weights[0] = weights[0] * c
    weights[-1] = weights[-1] / c
    return weights

## Network Architecture and Data Generation

Standard deep linear network with MSE loss. Data generated from a random target
matrix $W^*$ so the optimal solution is $W_3 \cdots W_0 = W^*$.

In [ ]:
def forward(weights, X):
    out = X.copy()
    for W in weights:
        out = W @ out
    return out

In [ ]:
def compute_loss(weights, X, Y):
    pred = forward(weights, X)
    return 0.5 * np.mean(np.sum((pred - Y)**2, axis=0))

In [ ]:
def compute_gradients(weights, X, Y):
    L = len(weights)
    N = X.shape[1]
    acts = [X.copy()]
    for W in weights:
        acts.append(W @ acts[-1])
    delta = (acts[-1] - Y) / N
    grads = [None] * L
    for l in range(L - 1, -1, -1):
        grads[l] = delta @ acts[l].T
        if l > 0:
            delta = weights[l].T @ delta
    return grads

In [ ]:
def make_data(seed):
    rng = np.random.RandomState(seed)
    W_target = rng.randn(DIM, DIM) * 0.5
    X = rng.randn(DIM, BATCH_SIZE) * 0.3
    Y = W_target @ X
    return X, Y

## Training Loop: Muon vs NormSGD

A unified training function handles both optimizers:
- **Muon** (`opt='muon'`): Applies Newton-Schulz to each gradient, then accumulates
  with momentum. The update is orthogonal (all SVs = 1).
- **NormSGD** (`opt='normsgd'`): Divides each gradient by its Frobenius norm, then
  accumulates with momentum. The update has normalized magnitude but retains the
  original gradient's *direction* (SV structure unchanged, just scaled to unit norm).

The key insight: both methods produce updates of comparable magnitude, but Muon's
updates have equalized SVs while NormSGD preserves the gradient's anisotropic SV
distribution. The loss ratio between them isolates the value of SV equalization.

In [ ]:
def train(w0, X, Y, lr, opt):
    weights = [W.copy() for W in w0]
    mom = [np.zeros_like(W) for W in weights]
    for step in range(NUM_STEPS):
        if compute_loss(weights, X, Y) > 1e10:
            return float('inf')
        grads = compute_gradients(weights, X, Y)
        for i in range(N_LAYERS):
            if opt == 'muon':
                mom[i] = MOMENTUM * mom[i] + newton_schulz(grads[i])
            elif opt == 'normsgd':
                nrm = np.linalg.norm(grads[i], 'fro')
                mom[i] = MOMENTUM * mom[i] + grads[i] / max(nrm, 1e-15)
            weights[i] -= lr * mom[i]
    return compute_loss(weights, X, Y)

## Main Experiment: Directional Advantage vs Balance Ratio

For each $c$ value:
1. Sweep LR for both Muon and NormSGD (using first 3 seeds for selection)
2. Evaluate both methods on all 5 seeds with the best LR
3. Compute the residual directional advantage = NormSGD loss / Muon loss
4. Also measure the actual weight norm ratio to confirm the prescribed balance

The final analysis computes the log-log correlation between $c$ and the residual
advantage. A strong negative correlation ($r < -0.5$) confirms that balance is
a reliable predictor of when Muon's direction matters.

In [ ]:
def main():
    seeds = [42 + i * 137 for i in range(NUM_SEEDS)]

    print("=" * 100)
    print("H21b: BALANCE RATIO PREDICTS DIRECTIONAL VALUE")
    print("=" * 100)
    print(f"c values: {C_VALUES}")
    print()

    results = {}

    for c in C_VALUES:
        print(f"\n  c={c:.1f}")
        best = {}

        for opt, grid in [('muon', LR_MUON), ('normsgd', LR_NORM)]:
            best_lr, best_loss = grid[-1], float('inf')
            for lr in grid:
                losses = [train(init_weights(s+5000, c), *make_data(s), lr, opt)
                          for s in seeds[:3]]
                ml = np.mean([l for l in losses if np.isfinite(l)] or [float('inf')])
                if ml < best_loss:
                    best_loss = ml
                    best_lr = lr
            best[opt] = best_lr

        # Full eval
        muon_losses = [train(init_weights(s+5000, c), *make_data(s), best['muon'], 'muon')
                       for s in seeds]
        norm_losses = [train(init_weights(s+5000, c), *make_data(s), best['normsgd'], 'normsgd')
                       for s in seeds]

        muon_mean = np.mean([l for l in muon_losses if np.isfinite(l)] or [float('inf')])
        norm_mean = np.mean([l for l in norm_losses if np.isfinite(l)] or [float('inf')])

        residual_adv = norm_mean / max(muon_mean, 1e-30)
        balance_ratio = c

        # Also measure actual layer norm ratios
        w = init_weights(seeds[0] + 5000, c)
        norms = [np.linalg.norm(W, 'fro') for W in w]
        actual_ratio = max(norms) / min(norms)

        results[c] = {
            'muon': muon_mean, 'normsgd': norm_mean,
            'residual_adv': residual_adv, 'actual_ratio': actual_ratio,
        }
        print(f"    Muon={muon_mean:.4e}  NormSGD={norm_mean:.4e}  "
              f"Residual directional adv: {residual_adv:.1f}x  "
              f"Actual ||W||_F ratio: {actual_ratio:.1f}")

    # =========================================================================
    # RESULTS
    # =========================================================================
    print(f"\n\n{'='*100}")
    print("RESULTS: RESIDUAL DIRECTIONAL ADVANTAGE vs BALANCE")
    print(f"{'='*100}")

    print(f"\n  {'c':>6}  {'||W|| ratio':>12}  {'Residual adv':>14}  {'Direction matters?':>20}")
    print("  " + "-" * 56)
    c_thresh = None
    for c in C_VALUES:
        r = results[c]
        matters = "YES" if r['residual_adv'] > 2.0 else "MARGINAL" if r['residual_adv'] > 1.2 else "NO"
        if c_thresh is None and r['residual_adv'] < 2.0:
            c_thresh = c
        print(f"  {c:>6.1f}  {r['actual_ratio']:>12.1f}  {r['residual_adv']:>14.1f}x  {matters:>20}")

    print(f"\n  Threshold c (directional advantage < 2x): {c_thresh if c_thresh else '>100'}")

    # Correlation
    cs = np.array(C_VALUES)
    advs = np.array([results[c]['residual_adv'] for c in C_VALUES])
    log_cs = np.log(cs)
    log_advs = np.log(np.clip(advs, 1e-10, None))
    r = np.corrcoef(log_cs, log_advs)[0, 1]
    print(f"  Correlation log(c) vs log(residual_adv): r = {r:.3f}")

    t1 = r < -0.5
    print(f"\n  T1: Negative correlation (r < -0.5)?  --> {'PASS' if t1 else 'FAIL'}  (r={r:.3f})")
    t2 = results[1.0]['residual_adv'] > 3 * results[100.0]['residual_adv']
    print(f"  T2: c=1 adv > 3x c=100 adv?           --> {'PASS' if t2 else 'FAIL'}")

    print(f"\n{'='*100}")
    print("EXPERIMENT COMPLETE")
    print(f"{'='*100}")

In [ ]:
if __name__ == '__main__':
    main()

## Conclusions

### What This Experiment Reveals

This experiment provides a **mechanistic explanation** for the crossover observed in H21a.
By using NormSGD as a control that matches Muon's scale normalization but not its
directional quality, we isolate exactly when and why the polar factor matters.

### Interpretation Guide

- **T1 PASS (strong negative correlation)**: Layer balance is a reliable predictor of
  directional value. Practitioners can use the weight norm ratio as a diagnostic:
  if $\max_\ell \|W_\ell\|_F / \min_\ell \|W_\ell\|_F \ll c_{\text{thresh}}$, Muon's
  directional advantage is active.

- **T2 PASS (3x reduction from $c=1$ to $c=100$)**: Confirms that the directional
  advantage is dramatically weaker under imbalance, consistent with the hypothesis
  that SV equalization operates on noise when gradients are scale-dominated.

- **Both fail**: Directional advantage may be independent of balance, suggesting
  the polar factor provides value through a different mechanism (e.g., curvature
  adaptation rather than SV equalization).

### Key Threshold

The $c_{\text{thresh}}$ where directional advantage drops below 2x is a practical
number: it tells practitioners at what balance ratio Muon's advantage reduces to
"merely" scale normalization. Networks with weight norms below this threshold
are in the "direction regime" where Muon's full gauge-fixing mechanism is active.

### Connection to Other Experiments

- **H21a**: Established the crossover; this experiment explains it mechanistically
- **H3a**: Per-SV gradient utilization -- related measure of directional quality
- **H6b**: Residual directional advantage at $c = 1$ -- the extreme balanced case
- **H15a**: Direction quality metric -- alternative way to measure the same phenomenon